# 🧠 Lasmoid 100M — Production Training (Kaggle T4)

**Full 3-phase pipeline: Pretrain → SFT → GRPO → Push to HuggingFace**

Optimized for Kaggle 2×T4 (30h/week). Aggressive checkpointing + resume support.

### Training Plan (15K pretrain steps):
| Phase | Steps | Dataset | Time (T4) |
|-------|-------|---------|------------|
| Pretrain | 12,000 | FineWeb-Edu 10BT | ~22h |
| SFT | 2,000 | OpenHermes 2.5 + GSM8K + MATH | ~4h |
| GRPO | 1,000 | Self-play w/ reasoning rewards | ~3h |

### Datasets (best for 100M):
- **Pretrain**: FineWeb-Edu (curated educational web text, 10B token sample)
- **SFT**: OpenHermes 2.5 (high-quality instruction data) + GSM8K (math reasoning)
- **GRPO**: Self-generated completions scored by reasoning quality reward

### Weights saved to: `Theory903/lasmoid-100m`

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1: Install & Setup
# ═══════════════════════════════════════════════════════════════════
!pip install -q torch transformers safetensors tiktoken numpy tqdm \
    datasets accelerate huggingface_hub

import os, sys, json, time, math

import torch, warnings
if not hasattr(torch, 'bf16'):
    torch.bf16 = torch.bfloat16
warnings.filterwarnings('ignore', message='.*incorrect regex pattern.*')
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

# ── Environment ──
IN_KAGGLE = os.path.exists('/kaggle')
WORK_DIR = '/kaggle/working/Lasmoid' if IN_KAGGLE else '/content/Lasmoid'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', '❌ No GPU! Enable GPU in Settings.'

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)')
print(f'✅ PyTorch {torch.__version__} | CUDA {torch.version.cuda}')

# ── HuggingFace Auth ──
HF_USER = 'Theory903'
HF_REPO = f'{HF_USER}/lasmoid-100m'
print(f'📦 Weights will be saved to: https://huggingface.co/{HF_REPO}')
print(f'\n⚠️  Run this in the next cell to authenticate:')
print(f'    from huggingface_hub import login; login()')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2: HuggingFace Login
# ═══════════════════════════════════════════════════════════════════
from huggingface_hub import login, HfApi, create_repo

# Option A: Use Kaggle secret (recommended)
# Add your HF token as a Kaggle secret named 'HF_TOKEN'
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('✅ Logged in via Kaggle secret')
except:
    # Option B: Interactive login
    login()

# Create the repo if it doesn't exist
api = HfApi()
try:
    create_repo(HF_REPO, exist_ok=True, private=False)
    print(f'✅ Repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'Repo creation: {e} (may already exist — continuing)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3: Clone/Upload Lasmoid Codebase
# ═══════════════════════════════════════════════════════════════════
import os, sys, shutil
from pathlib import Path

# Reset to kaggle working directory absolutely
os.chdir('/kaggle/working')

REPO_URL = 'https://github.com/Theory903/Lasmoid.git'
REPO_DIR = Path('/kaggle/working/Lasmoid')

# Cleanup nested duplicates
nested_dir = REPO_DIR / 'Lasmoid'
while nested_dir.exists():
    print('🧹 Cleaning nested duplicate directory...')
    shutil.rmtree(nested_dir, ignore_errors=True)

# Clone or pull cleanly
if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL}...')
    os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
else:
    if not (REPO_DIR / '.git').exists():
        print(f'Directory {REPO_DIR} exists but is not a valid git repository. Re-cloning...')
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        os.system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    else:
        print('Updating codebase to latest main branch...')
        os.system(f'git -C {REPO_DIR} fetch --all')
        os.system(f'git -C {REPO_DIR} reset --hard origin/main')

# Add to sys.path in correct order (prepend)
sys.path = [str(REPO_DIR), os.path.join(str(REPO_DIR), 'inference'), os.path.join(str(REPO_DIR), 'train')] + [p for p in sys.path if p not in (str(REPO_DIR), os.path.join(str(REPO_DIR), 'inference'), os.path.join(str(REPO_DIR), 'train'))]

os.chdir(str(REPO_DIR))
print(f'✅ Lasmoid loaded from {REPO_DIR}')
print(f'✅ Current Working Directory: {os.getcwd()}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4: Build Model
# ═══════════════════════════════════════════════════════════════════
from dataclasses import fields
import transformers
from inference.model import Lasmoid, ModelArgs, compute_loss, compute_grpo_loss
from train.optimizer import Muon, build_optimizers, build_param_groups
from train.scheduler import WSDScheduler

# Load config_100m.json
with open('config_100m.json') as f:
    cfg = json.load(f)

valid = {f.name for f in fields(ModelArgs)}
args = ModelArgs(**{k: v for k, v in cfg.items() if k in valid})

# Tokenizer loading with robust fallbacks
enc = None
try:
    enc = transformers.AutoTokenizer.from_pretrained(WORK_DIR, use_fast=True, local_files_only=True)
except Exception as e:
    print(f"⚠️ Standard AutoTokenizer failed to load: {e}. Trying with fix_mistral_regex=True...")

if enc is None:
    try:
        enc = transformers.AutoTokenizer.from_pretrained(WORK_DIR, use_fast=True, fix_mistral_regex=True, local_files_only=True)
    except Exception as e:
        print(f"⚠️ AutoTokenizer with fix_mistral_regex=True failed: {e}. Falling back to PreTrainedTokenizerFast...")

if enc is None:
    try:
        enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR, fix_mistral_regex=True, local_files_only=True)
    except Exception as e:
        print(f"⚠️ PreTrainedTokenizerFast with fix_mistral_regex=True failed: {e}. Trying without flag...")
        try:
            enc = transformers.PreTrainedTokenizerFast.from_pretrained(WORK_DIR, local_files_only=True)
        except Exception as e_final:
            raise RuntimeError(f"❌ Failed to load tokenizer from '{WORK_DIR}'. Error: {e_final}")

args.vocab_size = max(args.vocab_size, len(enc))
EOS_ID = enc.eos_token_id or 1
SEQ_LEN = args.max_seq_len  # 512

# Build
torch.manual_seed(1234)
model = Lasmoid(args).to(DEVICE)
model.train()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\n🏗️  Lasmoid-100M built: {n_params:.1f}M params')
print(f'    dim={args.dim}, layers={args.n_layers}, heads={args.n_heads}')
print(f'    experts={args.n_routed_experts}, concepts={args.num_concepts}')
print(f'    seq_len={SEQ_LEN}, vocab={args.vocab_size}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5: Training Configuration (BEST for 100M on T4)
# ═══════════════════════════════════════════════════════════════════

# ── Pretrain ──
PRETRAIN_STEPS = 12000
PRETRAIN_BATCH = 4
PRETRAIN_GRAD_ACCUM = 4  # effective batch = 16
PRETRAIN_MUON_LR = 2e-3
PRETRAIN_ADAMW_LR = 3e-4

# ── SFT ──
SFT_STEPS = 2000
SFT_BATCH = 4
SFT_LR = 1e-4

# ── GRPO ──
GRPO_STEPS = 1000
GRPO_BATCH = 2
GRPO_GROUP_SIZE = 4
GRPO_LR = 5e-5

# ── Shared ──
MAX_GRAD_NORM = 1.0
SAVE_EVERY = 1000  # checkpoint every 1K steps
LOG_EVERY = 100
CKPT_DIR = '/kaggle/working/checkpoints' if IN_KAGGLE else './checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Resume support ──
RESUME_FROM = None  # Set to a checkpoint path to resume
# e.g., RESUME_FROM = '/kaggle/working/checkpoints/pretrain_step_5000.pt'

print('📋 Training plan:')
print(f'   Pretrain: {PRETRAIN_STEPS} steps, eff_batch={PRETRAIN_BATCH*PRETRAIN_GRAD_ACCUM}')
print(f'   SFT:      {SFT_STEPS} steps, batch={SFT_BATCH}')
print(f'   GRPO:     {GRPO_STEPS} steps, group={GRPO_GROUP_SIZE}')
print(f'   Total tokens: ~{PRETRAIN_STEPS * PRETRAIN_BATCH * PRETRAIN_GRAD_ACCUM * SEQ_LEN / 1e9:.2f}B')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6: Download & Prepare Pretraining Data (FineWeb-Edu)
# ═══════════════════════════════════════════════════════════════════
from datasets import load_dataset

print('📥 Downloading FineWeb-Edu (sample-10BT)...')
print('   This is the highest-quality web text dataset for pretraining.')
print('   Curated for educational value — much better than raw web crawl.')

# FineWeb-Edu: educational web text, 10B token sample
# Best available open dataset for pretraining small LLMs
fw = load_dataset(
    'HuggingFaceFW/fineweb-edu-score-2',
    split='train',
    streaming=True,  # Don't download all 10B tokens at once
)

# Tokenize and pack — we need ~50M tokens for 12K steps @ batch=16 × seq=512
TARGET_TOKENS = PRETRAIN_STEPS * PRETRAIN_BATCH * PRETRAIN_GRAD_ACCUM * SEQ_LEN * 1.1
TARGET_TOKENS = int(min(TARGET_TOKENS, 100_000_000))  # Cap at 100M for T4 RAM

print(f'   Target: {TARGET_TOKENS/1e6:.0f}M tokens')

all_tokens = []
pbar = tqdm(total=TARGET_TOKENS, unit='tok', desc='Tokenizing FineWeb-Edu')

for doc in fw:
    text = doc.get('text', '')
    if len(text) < 50:  # Skip very short docs
        continue
    tokens = enc.encode(text) + [EOS_ID]
    all_tokens.extend(tokens)
    pbar.update(len(tokens))
    if len(all_tokens) >= TARGET_TOKENS:
        break

pbar.close()

# Pack into fixed-length sequences and split
all_tokens = all_tokens[:len(all_tokens) - len(all_tokens) % SEQ_LEN]
data = torch.tensor(all_tokens, dtype=torch.long)
n_train = int(0.98 * len(data))  # 98% train, 2% val (plenty of data)
train_data = data[:n_train]
val_data = data[n_train:]

print(f'\n📊 Pretraining data ready:')
print(f'   Train: {len(train_data)/1e6:.1f}M tokens ({len(train_data)//SEQ_LEN:,} seqs)')
print(f'   Val:   {len(val_data)/1e6:.1f}M tokens')

del all_tokens  # Free RAM

def get_pretrain_batch():
    ix = torch.randint(len(train_data) - SEQ_LEN - 1, (PRETRAIN_BATCH,))
    x = torch.stack([train_data[i:i+SEQ_LEN] for i in ix]).to(DEVICE)
    y = torch.stack([train_data[i+1:i+SEQ_LEN+1] for i in ix]).to(DEVICE)
    return x, y, torch.ones_like(x, dtype=torch.float32)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7: Phase 1 — PRETRAIN on FineWeb-Edu
# ═══════════════════════════════════════════════════════════════════
from huggingface_hub import upload_file, upload_folder

# Optimizers
opts = build_optimizers(
    model, muon_lr=PRETRAIN_MUON_LR, adamw_lr=PRETRAIN_ADAMW_LR,
    weight_decay=0.1, ns_steps=5, eps=1e-8
)

# WSD Schedule (2% warmup, 90% stable, 8% decay)
warmup = int(0.02 * PRETRAIN_STEPS)
stable = int(0.90 * PRETRAIN_STEPS)
decay = PRETRAIN_STEPS - warmup - stable

scheduler = WSDScheduler(
    optimizers=opts,
    warmup_steps=warmup,
    stable_steps=stable,
    decay_steps=decay,
    base_lrs=[[g['lr'] for g in opt.param_groups] for opt in opts],
    min_lr_ratio=0.1,
)

# Resume if checkpoint exists
start_step = 0
if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    start_step = ckpt.get('step', 0) + 1
    print(f'🔄 Resumed from step {start_step}')
    del ckpt

print(f'\n🚀 PHASE 1: Pretraining [{start_step}→{PRETRAIN_STEPS}]')
print(f'   Dataset: FineWeb-Edu | Effective batch: {PRETRAIN_BATCH * PRETRAIN_GRAD_ACCUM}')
print('─' * 70)

model.train()
losses = []
best_val_loss = float('inf')
t0 = time.time()

for step in range(start_step, PRETRAIN_STEPS):
    lr_mult = scheduler.step(step)
    for opt in opts:
        opt.zero_grad(set_to_none=True)

    step_loss = 0.0
    for _ in range(PRETRAIN_GRAD_ACCUM):
        xb, yb, mask = get_pretrain_batch()
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            logits, mtp_logits, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
            loss = compute_loss(
                logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
                loss_mask=mask,
                moe_aux_loss=model.last_moe_loss,
                moe_aux_coeff=getattr(args, 'moe_aux_coeff', 1.0),
                token_concept_loss=model.last_token_concept_loss,
                ignore_index=-100,
            )
            if mtp_logits is not None:
                mtp_l = F.cross_entropy(mtp_logits.view(-1, args.vocab_size),
                                        yb[:, 1:].contiguous().view(-1), ignore_index=-100)
                loss = loss + 0.3 * mtp_l
            loss = loss + 0.01 * model.last_pred_loss
            loss = loss / PRETRAIN_GRAD_ACCUM
        loss.backward()
        step_loss += loss.item() * PRETRAIN_GRAD_ACCUM

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    for opt in opts:
        opt.step()
    losses.append(step_loss)

    # Logging
    if step % LOG_EVERY == 0 and step > start_step:
        elapsed = time.time() - t0
        tps = (step - start_step) * PRETRAIN_BATCH * PRETRAIN_GRAD_ACCUM * SEQ_LEN / elapsed
        avg = sum(losses[-LOG_EVERY:]) / len(losses[-LOG_EVERY:])
        mem = torch.cuda.max_memory_allocated() / 1e9
        eta_h = (PRETRAIN_STEPS - step) / ((step - start_step) / elapsed) / 3600 if step > start_step else 0
        print(f'  Step {step:5d} | Loss {avg:.3f} | LR {lr_mult:.3f} | '
              f'{tps:.0f} tok/s | {mem:.1f}GB | ETA {eta_h:.1f}h')

    # Checkpoint
    if step > 0 and step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CKPT_DIR, f'pretrain_step_{step}.pt')
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'loss': step_loss,
            'losses_history': losses[-1000:],
        }, ckpt_path)
        print(f'  💾 Saved: {ckpt_path}')

# Final pretrain save + push to HF
pretrain_path = os.path.join(CKPT_DIR, 'pretrain_final.pt')
torch.save(model.state_dict(), pretrain_path)
print(f'\n✅ Pretrain done! Final loss: {losses[-1]:.4f}')
print(f'   Total time: {(time.time()-t0)/3600:.1f}h')

# Upload pretrain weights to HuggingFace
try:
    api.upload_file(
        path_or_fileobj=pretrain_path,
        path_in_repo='pretrain/model.pt',
        repo_id=HF_REPO,
        commit_message=f'Pretrain checkpoint (step {PRETRAIN_STEPS}, loss {losses[-1]:.4f})',
    )
    print(f'  ☁️  Uploaded to {HF_REPO}/pretrain/model.pt')
except Exception as e:
    print(f'  ⚠️  HF upload failed: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8: Prepare SFT Data (OpenHermes 2.5 + GSM8K)
# ═══════════════════════════════════════════════════════════════════

print('📥 Downloading SFT datasets...')

# OpenHermes 2.5: best open instruction dataset (GPT-4 quality conversations)
hermes = load_dataset('teknium/OpenHermes-2.5', split='train', streaming=True)

# GSM8K: grade-school math (reasoning chains)
gsm = load_dataset('openai/gsm8k', 'main', split='train')

sft_data = []  # list of (token_ids, loss_mask) tuples

# ── Format OpenHermes conversations ──
n_hermes = 0
MAX_HERMES = 8000
for doc in tqdm(hermes, total=MAX_HERMES, desc='OpenHermes'):
    convs = doc.get('conversations', [])
    if len(convs) < 2:
        continue
    # Format as chatml
    prompt_parts = []
    response_parts = []
    for msg in convs:
        role = msg.get('from', 'human')
        text = msg.get('value', '')
        if role in ('human', 'user'):
            prompt_parts.append(f'<|im_start|>user\n{text}\n<|im_end|>\n')
        elif role in ('gpt', 'assistant'):
            response_parts.append(f'<|im_start|>assistant\n{text}\n<|im_end|>\n')
    
    prompt_text = ''.join(prompt_parts)
    response_text = ''.join(response_parts)
    if not prompt_text or not response_text:
        continue
    
    prompt_ids = enc.encode(prompt_text)
    response_ids = enc.encode(response_text) + [EOS_ID]
    full = prompt_ids + response_ids
    
    if len(full) > SEQ_LEN:
        full = full[:SEQ_LEN]
        split_idx = min(len(prompt_ids), SEQ_LEN - 1)
    else:
        split_idx = len(prompt_ids)
        full = full + [EOS_ID] * (SEQ_LEN - len(full))
    
    mask = [0.0] * split_idx + [1.0] * (SEQ_LEN - split_idx)
    # Zero out padding positions in mask
    for i in range(len(prompt_ids) + len(response_ids), SEQ_LEN):
        mask[i] = 0.0
    
    sft_data.append((full[:SEQ_LEN], mask[:SEQ_LEN]))
    n_hermes += 1
    if n_hermes >= MAX_HERMES:
        break

# ── Format GSM8K with reasoning structure ──
for example in tqdm(gsm, desc='GSM8K'):
    q = example['question']
    a = example['answer']
    prompt = f'<|im_start|>user\n{q}\nSolve step by step.\n<|im_end|>\n<|im_start|>assistant\n<think>\n'
    response = f'{a}\n</think>\n<|im_end|>\n'
    
    p_ids = enc.encode(prompt)
    r_ids = enc.encode(response) + [EOS_ID]
    full = p_ids + r_ids
    
    if len(full) > SEQ_LEN:
        continue  # Skip too-long examples
    
    split_idx = len(p_ids)
    pad_len = SEQ_LEN - len(full)
    full = full + [EOS_ID] * pad_len
    mask = [0.0] * split_idx + [1.0] * len(r_ids) + [0.0] * pad_len
    sft_data.append((full, mask))

print(f'\n📊 SFT data: {len(sft_data):,} sequences ({n_hermes} OpenHermes + {len(gsm)} GSM8K)')

def get_sft_batch():
    idx = torch.randint(len(sft_data), (SFT_BATCH,))
    x_list, m_list = [], []
    for i in idx:
        tokens, mask = sft_data[i.item()]
        x_list.append(torch.tensor(tokens, dtype=torch.long))
        m_list.append(torch.tensor(mask, dtype=torch.float32))
    x = torch.stack(x_list).to(DEVICE)
    y = torch.cat([x[:, 1:], torch.full((x.shape[0], 1), EOS_ID, dtype=torch.long, device=DEVICE)], dim=1)
    mask = torch.stack(m_list).to(DEVICE)
    return x, y, mask

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9: Phase 2 — SFT on OpenHermes + GSM8K
# ═══════════════════════════════════════════════════════════════════

sft_opts = build_optimizers(model, muon_lr=SFT_LR*10, adamw_lr=SFT_LR, weight_decay=0.01)
sft_sched = WSDScheduler(
    optimizers=sft_opts,
    warmup_steps=int(0.03 * SFT_STEPS),
    stable_steps=int(0.87 * SFT_STEPS),
    decay_steps=int(0.10 * SFT_STEPS),
    base_lrs=[[g['lr'] for g in opt.param_groups] for opt in sft_opts],
    min_lr_ratio=0.01,
)

print(f'\n🎓 PHASE 2: SFT [{SFT_STEPS} steps]')
print('─' * 70)

model.train()
sft_losses = []
t0 = time.time()

for step in range(SFT_STEPS):
    sft_sched.step(step)
    for opt in sft_opts:
        opt.zero_grad(set_to_none=True)
    
    xb, yb, mask = get_sft_batch()
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        logits, mtp_logits, _, _, rmaps, _, adjs, eprobs = model(xb, xb)
        loss = compute_loss(
            logits, yb, rmaps, [model.last_vq_loss], adjs, eprobs,
            loss_mask=mask, moe_aux_loss=model.last_moe_loss, ignore_index=-100,
        )
        if mtp_logits is not None:
            loss = loss + 0.3 * F.cross_entropy(
                mtp_logits.view(-1, args.vocab_size),
                yb[:, 1:].contiguous().view(-1), ignore_index=-100)
    
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    for opt in sft_opts:
        opt.step()
    sft_losses.append(loss.item())
    
    if step % LOG_EVERY == 0 and step > 0:
        avg = sum(sft_losses[-LOG_EVERY:]) / LOG_EVERY
        print(f'  SFT Step {step:4d}/{SFT_STEPS} | Loss {avg:.4f}')

sft_path = os.path.join(CKPT_DIR, 'sft_final.pt')
torch.save(model.state_dict(), sft_path)
print(f'\n✅ SFT done! Loss: {sft_losses[-1]:.4f} | Time: {(time.time()-t0)/60:.0f}min')

try:
    api.upload_file(path_or_fileobj=sft_path, path_in_repo='sft/model.pt',
                    repo_id=HF_REPO, commit_message=f'SFT final (loss {sft_losses[-1]:.4f})')
    print(f'  ☁️  Uploaded SFT weights to {HF_REPO}')
except Exception as e:
    print(f'  ⚠️  Upload failed: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10: Phase 3 — GRPO Alignment
# ═══════════════════════════════════════════════════════════════════
from train.grpo_stability import compute_group_advantages, safe_reward
from train.reward import reasoning_self_evolution_reward

grpo_opts = build_optimizers(model, muon_lr=GRPO_LR*10, adamw_lr=GRPO_LR, weight_decay=0.0)

print(f'\n🎯 PHASE 3: GRPO Alignment [{GRPO_STEPS} steps]')
print('─' * 70)

model.train()
reward_history = []
t0 = time.time()

for step in range(GRPO_STEPS):
    for opt in grpo_opts:
        opt.zero_grad(set_to_none=True)
    
    xb, _, _ = get_sft_batch()
    prompt_len = SEQ_LEN // 2
    prompts = xb[:GRPO_BATCH, :prompt_len]
    prompts_exp = prompts.repeat_interleave(GRPO_GROUP_SIZE, dim=0)
    
    # Generate
    model.eval()
    with torch.no_grad():
        gen = model.generate(prompts_exp, SEQ_LEN - prompt_len, temperature=1.0, pad_token=EOS_ID)
    model.train()
    
    # Score
    rewards = []
    for i in range(gen.shape[0]):
        text = enc.decode(gen[i, prompt_len:].tolist())
        rewards.append(safe_reward(text, reasoning_self_evolution_reward))
    rewards_t = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)
    reward_history.append(rewards_t.mean().item())
    
    advantages = compute_group_advantages(rewards_t, GRPO_GROUP_SIZE)
    
    # Old logprobs
    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        lo, *_ = model(gen, gen)
        old_lp = F.log_softmax(lo[:, :-1], dim=-1).gather(2, gen[:, 1:].unsqueeze(-1)).squeeze(-1)
    model.train()
    
    # Policy step
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        logits, *_ = model(gen, gen)
        mask = torch.zeros_like(gen[:, 1:], dtype=torch.float32)
        mask[:, prompt_len-1:] = 1.0
        total_loss, pol_loss, kl_loss = compute_grpo_loss(
            logits[:, :-1], gen[:, 1:], advantages, old_lp,
            loss_mask=mask, clip_eps=0.2, kl_coeff=0.01)
    
    if torch.isfinite(total_loss):
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        for opt in grpo_opts:
            opt.step()
    
    if step % 50 == 0 and step > 0:
        avg_r = sum(reward_history[-50:]) / len(reward_history[-50:])
        print(f'  GRPO {step:4d}/{GRPO_STEPS} | Reward {avg_r:.3f} | '
              f'Policy {pol_loss.item():.4f} | KL {kl_loss.item():.4f}')

print(f'\n✅ GRPO done! Avg reward: {sum(reward_history[-50:])/50:.3f} | '
      f'Time: {(time.time()-t0)/60:.0f}min')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11: Save Final Model to HuggingFace
# ═══════════════════════════════════════════════════════════════════
from safetensors.torch import save_file as safetensors_save

# Save in safetensors format (industry standard)
final_dir = os.path.join(CKPT_DIR, 'final')
os.makedirs(final_dir, exist_ok=True)

# Model weights
state_dict = model.state_dict()
safetensors_path = os.path.join(final_dir, 'model.safetensors')
safetensors_save(state_dict, safetensors_path)

# Config
config_out = os.path.join(final_dir, 'config.json')
with open(config_out, 'w') as f:
    json.dump(cfg, f, indent=2)

# Model card
readme = f"""---
license: apache-2.0
language:
- en
tags:
- lasmoid
- hybrid-transformer
- mamba
- moe
- concept-memory
---

# Lasmoid-100M

A 100M parameter hybrid concept transformer combining:
- **CSA/MLA Attention** (compressed sparse + multi-head latent)
- **Mamba-2/SSD** (parallel state space recurrence)
- **Manifold-Constrained Hyper-Connections** (Sinkhorn-projected residual mixing)
- **Grey-Box MoE** (6 routed + 1 shared expert + dense FFN)
- **Elastic Sparse Concept Memory** (RVQ + GVQ + CIF compressor)
- **Multi-Token Prediction** (t+1 and t+2)
- **Muon Optimizer** (Newton-Schulz orthogonalization)

## Training
- **Pretrain**: {PRETRAIN_STEPS} steps on FineWeb-Edu (~{PRETRAIN_STEPS*PRETRAIN_BATCH*PRETRAIN_GRAD_ACCUM*SEQ_LEN/1e9:.1f}B tokens)
- **SFT**: {SFT_STEPS} steps on OpenHermes 2.5 + GSM8K
- **GRPO**: {GRPO_STEPS} steps with reasoning self-evolution reward
- **Hardware**: Kaggle T4 GPU

## Architecture
- dim: {args.dim}, layers: {args.n_layers}, heads: {args.n_heads}
- vocab: {args.vocab_size}, seq_len: {SEQ_LEN}
- experts: {args.n_routed_experts} routed + {args.n_shared_experts} shared
- concepts: {args.num_concepts}, codebook: {args.codebook_size}

## Usage
```python
from safetensors.torch import load_file
state_dict = load_file('model.safetensors')
model = Lasmoid(ModelArgs(**config)).eval()
model.load_state_dict(state_dict)
```
"""

with open(os.path.join(final_dir, 'README.md'), 'w') as f:
    f.write(readme)

# Upload entire folder to HuggingFace
print(f'\n☁️  Uploading to https://huggingface.co/{HF_REPO} ...')
try:
    api.upload_folder(
        folder_path=final_dir,
        repo_id=HF_REPO,
        commit_message=f'Lasmoid-100M final (pretrain+SFT+GRPO)',
    )
    print(f'\n🎉 SUCCESS! Model uploaded to: https://huggingface.co/{HF_REPO}')
    print(f'   Files: model.safetensors, config.json, README.md')
except Exception as e:
    print(f'\n❌ Upload failed: {e}')
    print(f'   Weights saved locally at: {final_dir}')
    print(f'   Upload manually: huggingface-cli upload {HF_REPO} {final_dir}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12: Generate Text (Verify Quality)
# ═══════════════════════════════════════════════════════════════════
from inference.sampler import full_sample

model.eval()

def generate(prompt, max_tokens=150, temperature=0.7, min_p=0.05):
    tokens = enc.encode(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=DEVICE)
    generated = list(tokens)
    
    pad_len = SEQ_LEN - len(tokens)
    if pad_len > 0:
        idx_padded = torch.cat([torch.full((1, pad_len), EOS_ID, dtype=torch.long, device=DEVICE), idx], dim=1)
    else:
        idx_padded = idx[:, -SEQ_LEN:]
    
    with torch.no_grad():
        logits, *_ = model(idx_padded, idx_padded, start_pos=0)
    
    gen = torch.Generator(device='cpu').manual_seed(42)
    for i in range(max_tokens):
        next_logits = logits[:, -1, :].cpu()
        next_id = full_sample(next_logits, generated, temperature=temperature, min_p=min_p, generator=gen)
        tid = next_id.item()
        if tid == EOS_ID:
            break
        generated.append(tid)
        
        inp = torch.tensor([[tid]], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            logits, *_ = model(x_enc=None, x_dec=inp, start_pos=SEQ_LEN + i)
    
    return enc.decode(generated[len(tokens):])

# Test prompts
prompts = [
    'Once upon a time, in a small village,',
    'The theory of relativity states that',
    'To solve this equation, first we need to',
    'The benefits of exercise include',
]

print('\n📝 Generation Samples:')
print('═' * 60)
for p in prompts:
    out = generate(p)
    print(f'\nPrompt: "{p}"')
    print(f'Output: {out[:300]}')
    print('─' * 60)

---
## 📋 Resume Guide (if session disconnects)

1. Re-run cells 1-4 (setup, login, clone, build model)
2. Set `RESUME_FROM` in Cell 5:
```python
RESUME_FROM = '/kaggle/working/checkpoints/pretrain_step_5000.pt'
```
3. Re-run Cell 7 — it will resume from the saved step

### Alternative: Download checkpoint from HuggingFace
```python
from huggingface_hub import hf_hub_download
path = hf_hub_download('Theory903/lasmoid-100m', 'pretrain/model.pt')
model.load_state_dict(torch.load(path, map_location='cuda'))
```